# Intro to Model and Data Drift

When a model is put into production, it moves **from a static environment** where the relationship between the input and output doesn't change to a **live environment** where the relationship between the input and output does change. The changing relationship causes the model to start losing predictive power. This loss in predictive power is called model drift.


In general, there are two reasons why drift can occur in Machine Learning models:

1. When an **external event mixes up the data**: customers' preferences suddenly change due to the pandemic, or a competitor launches a new product or service. If this has happened, an **update or retraining is necessary** for your model to keep producing accurate predictions.

2. When there is **something wrong with the data integrity**. For example, the model swaps two values like the height of a person and their age, or a website suddenly accepts blank fields due to a bug.

Both of these reasons are important to monitor and detect, but let's focus on the first one, the external events that change the data.

**An example:**

You've built a model that tries to predict how much a person will spend in an online webshop, using the age and the gender of a person. The training data set mostly consists of females, with the average age of all customers being 35. The model learns that mostly older people and women buy something more often. After monitoring the model's predictive performance, you notice that it's starting to decrease... But why? 

After doing some research you discovered that the webshop now has younger customers: the average age has dropped from 35 years to 25 years. Furthermore, the live data shows that currently not the women are majority, but the men!

So, the original relationship between the features that the model has learned from the original training data set isn't valid anymore on the real-time incoming data, which causes the model to lose its predictive power. **This loss in predictive power is called model drift and the change in the data is called data drift**.

Let's have a look at the NY Taxi dataset.

In [ ]:
import os
from dotenv import load_dotenv

import pandas as pd
import matplotlib.pyplot as plt
import mlflow



Let's start by loading the datasets. The first one is the dataset we train the model on:

In [ ]:
year = 2024
month = 1
color = "green"

df_24_1 = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

The second one is the dataset we compare the first one to:

In [ ]:
year = 2025
month = 2
color = "green"

if not os.path.exists(f"./data/{color}_tripdata_{year}-{month:02d}.parquet"):
    os.system(f"wget -P ./data https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet")

df_25_2 = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

We will only look at the features we trained the model on:

In [ ]:
features = ["PULocationID", "DOLocationID", "trip_distance", "passenger_count", "fare_amount", "total_amount"]
target = 'duration'

In [ ]:
def calculate_trip_duration_in_minutes(df):
    df["duration"] = (df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]).dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
    df = df[(df['passenger_count'] > 0) & (df['passenger_count'] < 8)]
    df = df[features + [target]]
    return df

In [ ]:
df_24_1 = calculate_trip_duration_in_minutes(df_24_1)
df_25_2 = calculate_trip_duration_in_minutes(df_25_2)
if len(df_24_1) < len(df_25_2):
    print("Trimming df_25_2 to match length of df_24_1")
    # randomly sample df_25_2
    df_25_2.sample(frac=1).reset_index(drop=True)
    df_25_2 = df_25_2.iloc[:len(df_24_1)]

elif len(df_24_1) > len(df_25_2):
    print("Trimming df_24_1 to match length of df_25_2")
    # randomly sample df_24_1
    df_24_1.sample(frac=1).reset_index(drop=True)
    df_24_1 = df_24_1.iloc[:len(df_25_2)]
else:
    print("Both dataframes have the same length")

### Now let's compare the data

In [ ]:
df_24_1.describe()

In [ ]:
df_25_2.describe()

In [ ]:
fig, ax = plt.subplots(1, len(df_24_1.columns), figsize=(3*len(df_24_1.columns), 3))
for i, var in enumerate(df_24_1):
    df_24_1[var].hist(ax=ax[i])
    ax[i].set_title(var)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, len(df_25_2.columns), figsize=(3*len(df_25_2.columns), 3))
for i, var in enumerate(df_25_2):
    df_25_2[var].hist(ax=ax[i])
    ax[i].set_title(var)
plt.show()

We see small changes in nearly all features. But we can also calculate the difference between the two datasets with the **Wasserstein distance for numerical features** and the **Jensen-Shannon distance for categorical features**. Both are metrics that calculate the distance between two probability distributions. The lower the value, the more similar the two distributions are.

Let's calculate the Wasserstein distance between the features of the two datasets:

In [ ]:
from scipy.stats import wasserstein_distance
from scipy.spatial.distance import jensenshannon

def drift_wasserstein_test(var_train: pd.Series, var_test: pd.Series):
    return wasserstein_distance(var_train, var_test)

def drift_jensenshannon_test(var_train: pd.Series, var_test: pd.Series):
    return jensenshannon(var_train, var_test)

In [ ]:
df_drift = pd.DataFrame(columns=["feature", "distance", "type"])

cat_features = ["PULocationID", "DOLocationID"]
num_features = ["trip_distance", "passenger_count", "fare_amount", "total_amount"]

for feature in num_features:
    df_drift = pd.concat(
        [df_drift, 
         pd.DataFrame(
            [{"feature": feature, 
            "distance": drift_wasserstein_test(df_24_1[feature],df_25_2[feature]),
            "type": "Wasserstein"}])],
            ignore_index=True
        )

for feature in cat_features:
    df_drift = pd.concat(
        [df_drift, 
         pd.DataFrame(
            [{"feature": feature, 
            "distance": drift_jensenshannon_test(df_24_1[feature],df_25_2[feature]),
            "type": "Jensen-Shannon"}])],
            ignore_index=True
        )

There are more **tests to detect drift**, but this is a good start. If you want to learn more about **drift detection**, check out this [article](https://www.evidentlyai.com/blog/data-drift-detection-large-datasets). For both Wasserstein and Jensen-Shannon distance there is a **threshold value**. If the distance is higher than the threshold, we can say that there is drift. For both the threshold is normally at 0.1. Let's check if there is drift in the data:

In [ ]:
df_drift['True/False Drift'] = df_drift['distance'] > 0.1
df_drift

So it seems we can confirm what we saw in the histograms. There is drift in nearly all features except the passenger count. Using this method **we can also detect drift in the target variable**. Let's have a look at the target variable:

In [ ]:
drift_wasserstein_test(df_24_1[target], df_25_2[target])

So we also have a drift in our target variable.

Of course you can also use the drift detection methods to detect drift in the prediction of our data. Let's load the model and do predictions on the two data sets:

In [ ]:
# Load environment variables from .env file
load_dotenv()
# Get the MLflow tracking URI from environment variables
MLFLOW_TRACKING_URI=os.getenv("MLFLOW_TRACKING_URI")

# Set up the connection to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
# Setup the MLflow experiment 
mlflow.set_experiment("green-taxi-monitoring")


gcp_key = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = gcp_key

In [ ]:
def load_model(model_name):
    alias = "production"
    model_uri = f"models:/{model_name}@{alias}"
    model = mlflow.pyfunc.load_model(model_uri)
    return model

In [ ]:
model_name = "green-taxi-ride-duration"

loaded_model = load_model(model_name)

In [ ]:
df_24_1["prediction"] = loaded_model.predict(df_24_1[features])

In [ ]:
df_25_2["prediction"] = loaded_model.predict(df_25_2[features])

And now we can calculate the drift in the predictions:

In [ ]:
drift_wasserstein_test(df_24_1["prediction"], df_25_2["prediction"])

And also here we can see the drift. All this drifts lead to a loss of predictive power. So we need to retrain our model. But how often do we need to retrain our model? This is a difficult question to answer. It depends on the problem you are trying to solve. If you are trying to predict the stock market, you probably need to retrain your model every day. If you are trying to predict the price of a house, you probably need to retrain your model every month. But in general, you need to **retrain your model when the drift is higher than the threshold** you set.

In the next notebook we will have a look at how we can do this in a more automated way.